In [2]:
!pip install pymupdf scikit-learn pandas numpy matplotlib


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Users\Lenovo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import os
import fitz
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [4]:
def extract_text(pdf_path):
    text = ""

    try:
        doc = fitz.open(pdf_path)

        for page in doc:
            text += page.get_text()

        doc.close()

    except Exception as e:
        print("Error:", pdf_path)

    return text

In [6]:
import os

print(os.listdir("data")[:10])

['ACCOUNTANT', 'ADVOCATE', 'AGRICULTURE', 'APPAREL', 'ARTS', 'AUTOMOBILE', 'AVIATION', 'BANKING', 'BPO', 'BUSINESS-DEVELOPMENT']


In [7]:
root = "data"

data = []

for category in os.listdir(root):

    category_path = os.path.join(root, category)

    if not os.path.isdir(category_path):
        continue

    print("Processing:", category)

    for file in os.listdir(category_path):

        if file.endswith(".pdf"):

            pdf_path = os.path.join(category_path, file)

            text = extract_text(pdf_path)

            if len(text.strip()) > 50:

                data.append({
                    "resume": text,
                    "label": category
                })

df = pd.DataFrame(data)

print(df.shape)
df.head()

Processing: ACCOUNTANT
Processing: ADVOCATE
Processing: AGRICULTURE
Processing: APPAREL
Processing: ARTS
Processing: AUTOMOBILE
Processing: AVIATION
Processing: BANKING
Processing: BPO
Processing: BUSINESS-DEVELOPMENT
Processing: CHEF
Processing: CONSTRUCTION
Processing: CONSULTANT
Processing: DESIGNER
Processing: DIGITAL-MEDIA
Processing: ENGINEERING
Processing: FINANCE
Processing: FITNESS
Processing: HEALTHCARE
Processing: HR
Processing: INFORMATION-TECHNOLOGY
Processing: PUBLIC-RELATIONS
Processing: SALES
Processing: TEACHER
(2483, 2)


,resume,label
0,ACCOUNTANT\nSummary\nFinancial Accountant spec...,ACCOUNTANT
1,STAFF ACCOUNTANT\nSummary\nHighly analytical a...,ACCOUNTANT
2,ACCOUNTANT\nProfessional Summary\nTo obtain a ...,ACCOUNTANT
3,SENIOR ACCOUNTANT\nExperience\nCompany Name Ju...,ACCOUNTANT
4,SENIOR ACCOUNTANT\nProfessional Summary\nSenio...,ACCOUNTANT


In [8]:
df["label"].value_counts()

label
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      119
ACCOUNTANT                118
ADVOCATE                  118
FINANCE                   118
ENGINEERING               118
CHEF                      118
FITNESS                   117
AVIATION                  117
SALES                     116
HEALTHCARE                115
CONSULTANT                115
BANKING                   115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64

In [9]:
df.shape

(2483, 2)

In [10]:
df["label"].nunique()

24

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    df["resume"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [16]:
model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=30000,
            stop_words="english",
            ngram_range=(1,2),
            min_df=2
        )
    ),
    (
        "clf",
        LogisticRegression(
            max_iter=5000,
            n_jobs=-1,
            class_weight="balanced"
        )
    )
])

model.fit(X_train, y_train)

,steps,"[('tfidf', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [17]:
pred=model.predict(X_test)

acc = accuracy_score(y_test, pred)

print("Accuracy:", round(acc * 100,2),"%")

Accuracy: 65.59 %


In [18]:
print(classification_report(y_test, pred))

                        precision    recall  f1-score   support

            ACCOUNTANT       0.69      0.83      0.75        24
              ADVOCATE       0.67      0.50      0.57        24
           AGRICULTURE       0.73      0.62      0.67        13
               APPAREL       0.69      0.58      0.63        19
                  ARTS       0.50      0.29      0.36        21
            AUTOMOBILE       0.50      0.14      0.22         7
              AVIATION       0.85      0.71      0.77        24
               BANKING       0.59      0.57      0.58        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.42      0.71      0.53        24
                  CHEF       0.89      0.71      0.79        24
          CONSTRUCTION       0.79      0.68      0.73        22
            CONSULTANT       0.40      0.17      0.24        23
              DESIGNER       0.82      0.67      0.74        21
         DIGITAL-MEDIA       0.61      

C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_c

In [19]:
df["resume"].str.len().describe()

count     2483.000000
mean      5940.546919
std       2625.906454
min        692.000000
25%       4831.000000
50%       5557.000000
75%       6866.500000
max      35132.000000
Name: resume, dtype: float64

In [20]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=30000,
            stop_words="english",
            ngram_range=(1,2),
            min_df=2
        )
    ),
    (
        "clf",
        LinearSVC()
    )
])

model.fit(X_train, y_train)

pred = model.predict(X_test)

from sklearn.metrics import accuracy_score

print(
    "Accuracy:",
    round(accuracy_score(y_test, pred) * 100, 2),
    "%"
)

Accuracy: 70.62 %


In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score

rf_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=10000,
            stop_words="english"
        )
    ),
    (
        "clf",
        RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )
    )
])

rf_model.fit(X_train, y_train)

pred = rf_model.predict(X_test)

print(
    "Accuracy:",
    round(accuracy_score(y_test, pred) * 100, 2),
    "%"
)

Accuracy: 64.99 %


In [22]:
!pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   -- ------------------------------------- 7.1/101.7 MB 39.6 MB/s eta 0:00:03
   ----- ---------------------------------- 14.7/101.7 MB 38.4 MB/s eta 0:00:03
   ------- -------------------------------- 19.4/101.7 MB 37.1 MB/s eta 0:00:03
   ------- -------------------------------- 19.4/101.7 MB 37.1 MB/s eta 0:00:03
   -------- ------------------------------- 22.3/101.7 MB 22.4 MB/s eta 0:00:04
   ----------- ---------------------------- 30.1/101.7 MB 24.5 MB/s eta 0:00:03
   ------------- -------------------------- 34.9/101.7 MB 25.7 MB/s eta 0:00:03
   ---------------- ----------------------- 41.4/101.7 MB 25.6 MB/s eta 0:00:03
   ------------------ --------------------- 46.9/101.7 MB 25.7 MB/s eta 0:00:03
   ------------------ --------------------- 48.2/101.7 MB 25.0 MB/s eta 0:00:03
   ------------------- -------------------- 49.0/101.7 MB 21.7 MB/s eta 0:00:03
   -------------------- ------------------- 51.1/1


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Users\Lenovo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [30]:
sample_resume = """
Experienced software engineer with expertise in Python,
JavaScript, React, Node.js, SQL, REST APIs and cloud deployment.
Worked on full-stack web applications and scalable systems.
"""

prediction = model.predict([sample_resume])

print("Predicted Category:", prediction[0])

Predicted Category: ENGINEERING


In [32]:
pdf_path = "data/INFORMATION-TECHNOLOGY/15118506.pdf"

text = extract_text(pdf_path)

prediction = model.predict([text])

print("Prediction:", prediction[0])

Prediction: INFORMATION-TECHNOLOGY


In [33]:
my_resume = extract_text("Shubham_Solanki_Resume.pdf")

prediction = model.predict([my_resume])

print("Predicted Category:", prediction[0])

Predicted Category: ENGINEERING


In [34]:
scores = model.decision_function([my_resume])

print(scores)


[[-1.13664867 -1.02392838 -0.76570682 -0.9556722  -0.66912339 -1.02402036
  -0.87859809 -0.98871139 -0.94847173 -1.10968886 -0.92784713 -0.97327377
  -0.92703339 -1.17209284 -0.6380437  -0.53016825 -1.08802301 -1.03726686
  -1.02273827 -0.99791669 -0.97782274 -1.04442499 -1.09252472 -0.97257691]]


In [35]:
classes = model.named_steps['clf'].classes_

scores = model.decision_function([my_resume])[0]

top5_idx = scores.argsort()[-5:][::-1]

for idx in top5_idx:
    print(classes[idx], ":", round(scores[idx], 4))

ENGINEERING : -0.5302
DIGITAL-MEDIA : -0.638
ARTS : -0.6691
AGRICULTURE : -0.7657
AVIATION : -0.8786
